This module aims to bulk download original reports from urls scraped and cleansed. Special notices include:
1. Set "headers" in GET requests as shown in your Browser (mimicking ordinary file download)
2. Set time lag/pause between requests and add randomness (imitating human behavior of file download)
3. Make sure VPN/proxy server is switched on when running the code



In [1]:
import os
import time
from datetime import datetime
import requests
from requests.adapters import HTTPAdapter
from urllib3.util import Retry
import pandas as pd
import random

# Check output_folder and continent_selected before running the code
continent_selected = "northamerica" # africa/asia/europe/northamerica/southamerica
output_folder = "../../data/original_report/" + ("usgs_" + continent_selected) + "/"

# Read-in merged URL list
url_path = "../../data/merged_url/usgs_merged.csv"
url_table = pd.read_csv(url_path)
url_table['year'] = pd.to_datetime(url_table['year'])

# url_list is a copy or slice of url_table (based on continent_selected)
url_list = url_table.query("continent == @continent_selected")

# Re-check
print(output_folder)
url_list

../../data/original_report/usgs_northamerica/


,year,country,url,continent
1850,2005-01-01,Canada,https://d9-wret.s3.us-west-2.amazonaws.com/ass...,northamerica
1851,2006-01-01,Canada,https://d9-wret.s3.us-west-2.amazonaws.com/ass...,northamerica
1852,2007-01-01,Canada,https://d9-wret.s3.us-west-2.amazonaws.com/ass...,northamerica
1853,2008-01-01,Canada,https://d9-wret.s3.us-west-2.amazonaws.com/ass...,northamerica
1854,2009-01-01,Canada,https://d9-wret.s3.us-west-2.amazonaws.com/ass...,northamerica
1855,2010-01-01,Canada,https://d9-wret.s3.us-west-2.amazonaws.com/ass...,northamerica
1856,2011-01-01,Canada,https://d9-wret.s3.us-west-2.amazonaws.com/ass...,northamerica
1857,2012-01-01,Canada,https://d9-wret.s3.us-west-2.amazonaws.com/ass...,northamerica
1858,2013-01-01,Canada,https://d9-wret.s3.us-west-2.amazonaws.com/ass...,northamerica
1859,2014-01-01,Canada,https://d9-wret.s3.us-west-2.amazonaws.com/ass...,northamerica


In [2]:
## Define environment for request.Session()
# Define headers
headers_strategy = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36 Edg/148.0.0.0'
}

# Define retry strategy
retry_strategy = Retry(
    total = 5,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504]
)
retry_strategy_adapter = HTTPAdapter(max_retries=retry_strategy)

## Initialize a log file
log_folder = "../../log/" # Folder where all files are stored
log_name = 'downloadfile_' + str(int(time.time())) + '.txt' # Both name of task and timestamp included
log_path = os.path.join(log_folder, log_name)
# Open the log before webspider begins to operate
with open(log_path, 'w', encoding='utf-8') as log: # IMPORTANT: Designate encoding as 'utf-8' to avoid error caused by unrecognizable symbols under 'gbk' encoding
    
    ## WEBSPIER SECTION
    with requests.Session() as session:
        # Update environment of the requests.Session() with parameters already defined
        session.headers.update(headers_strategy)
        session.mount(prefix='https://', adapter=retry_strategy_adapter)
        session.mount(prefix='http://', adapter=retry_strategy_adapter)

        counter = 0 # Initiate a counter for indicating download process
        for index in url_list.index: # Iterate through url_list
            counter = counter + 1 # Update the counter

            # Extract basic information about file to download from the row corresponding to the index
            year = str(url_list.loc[index]['year'].year)
            country = url_list.loc[index]['country']
            continent = url_list.loc[index]['continent']
            url = url_list.loc[index]['url']
            # Name the file to download and save ("[country]_[year].pdf")
            filename = '_'.join([country,year])+'.pdf'
            filepath = os.path.join(output_folder, filename)
            
            # Start the timer 
            start_time = time.perf_counter()
            start_time_legible = str(datetime.now())

            try:
                # Print and write into log the beginning of operation
                message_start = f"Downloading {counter}/{len(url_list.index)} [{country} in {year} from: {url}]"
                print(message_start)
                log.write(start_time_legible + '\n')
                log.write(message_start + '\n\n') # Add two line breaks

                ## CORE OF WEBSPIDER
                # Get request ()
                response = session.get(
                    url = url,
                    timeout = (5,30), # 5s connect timeout limit and 30s read timeout limit
                    stream = True # Temporarily save response to hard drive rather than RAM and prepare for iteration below
                )
                # Check whether the request is properly handled by the server (jump down to "except" if status_code is not 200 or there's a timeout)
                response.raise_for_status()
                # Write response into pdf-suffix file by chunk (chunk_size set as 8192 bytes or 8 KB based on convention)
                with open(filepath,'wb') as file:
                    for chunk in response.iter_content(chunk_size=8192):
                        file.write(chunk)

                # End the timer
                end_time = time.perf_counter()
                end_time_legible = str(datetime.now())
                execution_time = end_time - start_time # Calculate execution interval
                # Print and write to log the termination of operation
                message_end = f"Successfully saved to {filepath}. Execution time {execution_time:.2f} seconds"
                print(message_end)
                log.write(end_time_legible + '\n')
                log.write(message_end + '\n\n') # Add two line breaks

                ## CORE OF WEBSPIDER (CONTINUE): pause for random seconds (no more than 4s) before initiate another request 
                time.sleep(2 + random.random() * 2)
            
            except requests.exceptions.RequestException as error:
                # End the timer
                end_time = time.perf_counter()
                end_time_legible = str(datetime.now())
                execution_time = end_time - start_time
                # Print and write to log the termination of operation
                message_end = f"Failed to download {url}. Error: {error}. Execution time {execution_time:.2f} seconds"
                print(message_end)
                log.write(end_time_legible + '\n')
                log.write(message_end + '\n\n')

                ## CORE OF WEBSPIDER (CONTINUE): pause for random seconds (no more than 7s) before initiate another request 
                time.sleep(5 + random.random() * 2)

Successfully saved to ../../data/original_report/usgs_northamerica/Canada_2005.pdf. Execution time 1.69 seconds
Successfully saved to ../../data/original_report/usgs_northamerica/Canada_2006.pdf. Execution time 0.33 seconds
Successfully saved to ../../data/original_report/usgs_northamerica/Canada_2007.pdf. Execution time 0.43 seconds
Successfully saved to ../../data/original_report/usgs_northamerica/Canada_2008.pdf. Execution time 0.34 seconds
Successfully saved to ../../data/original_report/usgs_northamerica/Canada_2009.pdf. Execution time 0.36 seconds
Successfully saved to ../../data/original_report/usgs_northamerica/Canada_2010.pdf. Execution time 0.37 seconds
Successfully saved to ../../data/original_report/usgs_northamerica/Canada_2011.pdf. Execution time 0.34 seconds
Successfully saved to ../../data/original_report/usgs_northamerica/Canada_2012.pdf. Execution time 0.37 seconds
Successfully saved to ../../data/original_report/usgs_northamerica/Canada_2013.pdf. Execution time 0.34 